# XAI stability -- TESTS

In [ ]:
import sys

!{sys.executable} -m pip install nbimporter
!{sys.executable} -m pip install tensorflow
!{sys.executable} -m pip install torch

#!git clone https://github.com/AI4LIFE-GROUP/OpenXAI.git
!{sys.executable} -m pip install -e OpenXAI

In [2]:
import time
import numpy as np
import pandas as pd
import nbimporter

# Utils
import torch
import os
import pickle
from sklearn.base import clone

import xgboost as xgb
from sklearn.neural_network import MLPClassifier


import Taylor_Explainer as texp
import XAI_stability_metrics as stab


import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
import openxai

# Data loaders
from openxai.dataloader import return_loaders

# Perturbation methods required for the computation of the relative stability metrics
from openxai.explainers.catalog.perturbation_methods import NormalPerturbation
from openxai.explainers.catalog.perturbation_methods import NewDiscrete_NormalPerturbation

In [4]:
import warnings
warnings.filterwarnings("ignore", message='should_run_async')

# Perturbation definition

In [5]:
# Perturbation class parameters
perturbation_mean= 0.0
perturbation_std= 0.05
perturbation_flip_percentage= 0.01
    
perturbation= NormalPerturbation('tabular',
                                 mean=perturbation_mean,
                                 std_dev=perturbation_std,
                                 flip_percentage=perturbation_flip_percentage)

def generate_mask(explanation, top_k):
    mask_indices= torch.topk(explanation, top_k).indices
    mask= torch.zeros(explanation.shape) > 10
    for i in mask_indices:
        mask[i]= True
    return mask

# Loading data

# 1. Synthetic - 20 features - Numeric

In [6]:
ox_path= 'data/synth_OX_20/processed/'

train_ox= pd.read_csv(ox_path + 'X_train.csv')
test_ox = pd.read_csv(ox_path + 'X_test.csv')
labels_train_ox= pd.read_csv(ox_path + 'y_train.csv')
labels_test_ox = pd.read_csv(ox_path + 'y_test.csv')

In [7]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_ox= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn1_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn1_model_ox.predict(test_ox))
acc_nn1_ox

0.83

In [8]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_ox= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn2_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn2_model_ox.predict(test_ox))
acc_nn2_ox

0.83

In [9]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ox= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn3_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn3_model_ox.predict(test_ox))
acc_nn3_ox

0.84

In [10]:
# definitions ---- 154 samples from test dataset

# get n and m parameters from train and labels_train
n_ox, m_ox= texp.get_n_m_sizes(test_ox.loc[0:153], labels_test_ox[0:153])

# conversion of train_ox and labels_train_ox data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_ox.values)
tn_lb_tr= torch.from_numpy(labels_train_ox.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_ox= dict()

h_min_dist_ox= texp.get_minimum_distance(train_ox)

# T-Exp explanation settings
descriptor_ox['h_min']= h_min_dist_ox
descriptor_ox['h_max']= 1
descriptor_ox['jacobian_eps']= 1e-3
descriptor_ox['max_itr']= 30
descriptor_ox['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_ox['num_samples']= 30
descriptor_ox['num_perts']= 10   # RIS/ROS
descriptor_ox['pert_max_distance']= (h_min_dist_ox/2)
descriptor_ox['num_runs']= 10    # RES
descriptor_ox['feature_metadata']= ['c'] * n_ox
descriptor_ox['p_norm']= 2
descriptor_ox['eps_norm']= 1e-6
descriptor_ox['top_k']= 0
descriptor_ox['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_ox['top_k'])

In [46]:
# ---- 154 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_ox
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_ox= stab.relative_stability(nn1_model_ox, test_ox[0:153], labels_test_ox[0:153], perturbation, 
                                        descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_ox

RIS/ROS --
Progress: [████████████████████████████████████████] 153/153 Est wait 00:0.03


--- 7233.1 seconds ---


{'t_exp_ris_max': 6232459.282776426,
 'std(t_exp_ris_max)': 541928.6275944225,
 'shap_ris_max': 202623.02528666324,
 'std(shap_ris_max)': 29791.176166268735,
 'lime_ris_max': 1131.9958358893905,
 'std(lime_ris_max)': 99.5336783935745,
 't_exp_ris_mean': 11756.709204671091,
 'std(t_exp_ris_mean)': 104093.55208937744,
 'shap_ris_mean': 2949.012127427184,
 'std(shap_ris_mean)': 8727.41812496453,
 'lime_ris_mean': 7.487263028429098,
 'std(lime_ris_mean)': 29.509190131435897,
 't_exp_ros_max': 2400298092865.2676,
 'std(t_exp_ros_max)': 198942939632.29453,
 'shap_ros_max': 19166666818.081284,
 'std(shap_ros_max)': 1544307033.9760387,
 'lime_ros_max': 4775430.779914704,
 'std(lime_ros_max)': 490126.3546041153,
 't_exp_ros_mean': 2497444000.0722294,
 'std(t_exp_ros_mean)': 25162145037.97871,
 'shap_ros_mean': 30699269.27928684,
 'std(shap_ros_mean)': 368434399.3751729,
 'lime_ros_mean': 37049.03097270383,
 'std(lime_ros_mean)': 202371.17780664918,
 'shap_kernel_ris_max': 335.81792611530585,
 '

In [48]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ox
start= time.time()
print('RES --')
res_nn1_ox= stab.run_stability(nn1_model_ox, test_ox[0:153], labels_test_ox[0:153], 
                               descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_ox

RES --
Progress: [████████████████████████████████████████] 153/153 Est wait 00:0.04


--- 6646.14 seconds ---


{'texp_res': 9.455164053283262e-15,
 'shap_res': 0.15312519841256989,
 'shap_kernel_res': 0.05378987679156172,
 'shap_exact_res': '--',
 'lime_res': 5.801606246868625e-17,
 'itGd_res': 1.1149867635186251e-14,
 'iXGd_res': 8.145812e-06,
 'dLif_res': 8.302823e-06,
 'lwrp_res': 8.777078e-06,
 'smoothG_res': 7.23048052597422,
 'vanillaG_res': 1.3662861e-05,
 'GuidBprop_res': 1.3662861e-05,
 'occlusion_res': 5.2452087e-06}

In [50]:
# evaluate relative input/output stability -- using the 2 first instances from train_ox
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_ox= stab.relative_stability(nn2_model_ox, test_ox[0:153], labels_test_ox[0:153], perturbation, 
                                        descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_ox

RIS/ROS --
Progress: [████████████████████████████████████████] 153/153 Est wait 00:0.00


--- 13919.55 seconds ---


{'t_exp_ris_max': 5816.970207042285,
 'std(t_exp_ris_max)': 499.7208078421638,
 'shap_ris_max': 171070.8369154764,
 'std(shap_ris_max)': 22206.07607221791,
 'lime_ris_max': 296.82670073540544,
 'std(lime_ris_max)': 30.45997996419992,
 't_exp_ris_mean': 59.80724535235985,
 'std(t_exp_ris_mean)': 354.08858864864425,
 'shap_ris_mean': 2129.215893768227,
 'std(shap_ris_mean)': 9058.562143355168,
 'lime_ris_mean': 4.38482070337584,
 'std(lime_ris_mean)': 14.358616638807518,
 't_exp_ros_max': 90319.22508529491,
 'std(t_exp_ros_max)': 7414.282502297692,
 'shap_ros_max': 538734.4440415618,
 'std(shap_ros_max)': 53799.87369849377,
 'lime_ros_max': 3787.2942468042306,
 'std(lime_ros_max)': 386.147923298815,
 't_exp_ros_mean': 165.50918742343714,
 'std(t_exp_ros_mean)': 861.0355047448513,
 'shap_ros_mean': 3250.230342958256,
 'std(shap_ros_mean)': 13914.162922092042,
 'lime_ros_mean': 11.890020217139286,
 'std(lime_ros_mean)': 44.37037420889288,
 'shap_kernel_ris_max': 425.3970892947496,
 'std(sh

In [52]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ox
start= time.time()
print('RES --')
res_nn2_ox= stab.run_stability(nn2_model_ox, test_ox[0:153], labels_test_ox[0:153], 
                               descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_ox

RES --
Progress: [████████████████████████████████████████] 153/153 Est wait 00:0.05


--- 12472.55 seconds ---


{'texp_res': 4.0943002132167226e-15,
 'shap_res': 0.17656098450677027,
 'shap_kernel_res': 0.04678864931999716,
 'shap_exact_res': '--',
 'lime_res': 5.3617058706823765e-17,
 'itGd_res': 7.838739178637249e-15,
 'iXGd_res': 3.1411776e-06,
 'dLif_res': 3.1047741e-06,
 'lwrp_res': 2.6744574e-06,
 'smoothG_res': 5.829929827515715,
 'vanillaG_res': 7.0414253e-06,
 'GuidBprop_res': 7.0414253e-06,
 'occlusion_res': 2.311554e-06}

In [54]:
# evaluate relative input/output stability -- using the 2 first instances from train_ox
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_ox= stab.relative_stability(nn3_model_ox, test_ox[0:153], labels_test_ox[0:153], perturbation, 
                                        descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_ox

RIS/ROS --
Progress: [████████████████████████████████████████] 153/153 Est wait 00:0.09


--- 16052.32 seconds ---


{'t_exp_ris_max': 11302.936295024148,
 'std(t_exp_ris_max)': 915.5709368576528,
 'shap_ris_max': 139739.8475091104,
 'std(shap_ris_max)': 19761.20262359331,
 'lime_ris_max': 319.81994007735665,
 'std(lime_ris_max)': 25.930331699016442,
 't_exp_ris_mean': 40.30576677660633,
 'std(t_exp_ris_mean)': 179.43193761533067,
 'shap_ris_mean': 2069.42418196957,
 'std(shap_ris_mean)': 6493.183881899847,
 'lime_ris_mean': 2.0365003768026724,
 'std(lime_ris_mean)': 13.61353512704214,
 't_exp_ros_max': 817888.7474505965,
 'std(t_exp_ros_max)': 65888.15033539564,
 'shap_ros_max': 14685546.696755521,
 'std(shap_ros_max)': 1184135.6749119384,
 'lime_ros_max': 25260.786902376563,
 'std(lime_ros_max)': 2041.3968152130242,
 't_exp_ros_mean': 680.1615978480564,
 'std(t_exp_ros_mean)': 7028.712586207266,
 'shap_ros_mean': 16413.893120235392,
 'std(shap_ros_mean)': 150046.5137301343,
 'lime_ros_mean': 23.90853036507994,
 'std(lime_ros_mean)': 213.50935822582343,
 'shap_kernel_ris_max': 2391.302505135726,
 's

In [56]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ox
start= time.time()
print('RES --')
res_nn3_ox= stab.run_stability(nn3_model_ox, test_ox[0:153], labels_test_ox[0:153], 
                               descriptor_ox, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_ox

RES --
Progress: [████████████████████████████████████████] 153/153 Est wait 00:0.01


--- 17758.81 seconds ---


{'texp_res': 5.630757419431282e-15,
 'shap_res': 0.14188053709214507,
 'shap_kernel_res': 0.049825771838426625,
 'shap_exact_res': '--',
 'lime_res': 5.793820594181519e-17,
 'itGd_res': 5.208590938883218e-15,
 'iXGd_res': 4.5775437e-06,
 'dLif_res': 4.874844e-06,
 'lwrp_res': 4.572496e-06,
 'smoothG_res': 6.611081546656646,
 'vanillaG_res': 7.6779015e-06,
 'GuidBprop_res': 7.6779015e-06,
 'occlusion_res': 2.5788913e-06}

# TODO LIST
# - Setar parâmetros dos modelos de 2., 3., 8., 10.
# - Verificar h_min de todos os conjuntos antes de testar e fixar um h_mim, caso necessário

# DONE -- synth, diabetes, independent

# 2. Adult Income

In [6]:
ad_path= 'data/adult/processed/'

train_ad= pd.read_csv(ad_path + 'X_train.csv')
test_ad = pd.read_csv(ad_path + 'X_test.csv')
labels_train_ad= pd.read_csv(ad_path + 'y_train.csv')
labels_test_ad = pd.read_csv(ad_path + 'y_test.csv')

In [ ]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_ad= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_ad.fit(train_ad, labels_train_ad.values.ravel())

acc_nn1_ad= sklearn.metrics.accuracy_score(labels_test_ad.values.ravel(), nn1_model_ad.predict(test_ad))
acc_nn1_ad

In [ ]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_ad= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_ad.fit(train_ad, labels_train_ad.values.ravel())

acc_nn2_ad= sklearn.metrics.accuracy_score(labels_test_ad.values.ravel(), nn2_model_ad.predict(test_ad))
acc_nn2_ad

In [ ]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ad= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_ad.fit(train_ad, labels_train_ad.values.ravel())

acc_nn3_ad= sklearn.metrics.accuracy_score(labels_test_ad.values.ravel(), nn3_model_ad.predict(test_ad))
acc_nn3_ad

In [ ]:
# definitions ---- 624 samples from test dataset

# get n and m parameters from train and labels_train
n_ad, m_ad= texp.get_n_m_sizes(test_ad.loc[0:623], labels_test_ad[0:623])

# conversion of train_ad and labels_train_ad data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_ad.values)
tn_lb_tr= torch.from_numpy(labels_train_ad.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_ad= dict()

h_min_dist_ad= texp.get_minimum_distance(train_ad)

# T-Exp explanation settings
descriptor_ad['h_min']= h_min_dist_ad
descriptor_ad['h_max']= 1
descriptor_ad['jacobian_eps']= 1e-3
descriptor_ad['max_itr']= 30
descriptor_ad['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_ad['num_samples']= 30
descriptor_ad['num_perts']= 10   # RIS/ROS
descriptor_ad['pert_max_distance']= (h_min_dist_ad/2)
descriptor_ad['num_runs']= 10    # RES
descriptor_ad['feature_metadata']= ['c'] * n_ad
descriptor_ad['p_norm']= 2
descriptor_ad['eps_norm']= 1e-6
descriptor_ad['top_k']= 0
descriptor_ad['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_ad['top_k'])

In [ ]:
# ---- 624 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_ad
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_ad= stab.relative_stability(nn1_model_ad, test_ad[0:623], labels_test_ad[0:623], perturbation, 
                                        descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_ad

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ad
start= time.time()
print('RES --')
res_nn1_ad= stab.run_stability(nn1_model_ad, test_ad[0:623], labels_test_ad[0:623], 
                               descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_ad

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_ad
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_ad= stab.relative_stability(nn2_model_ad, test_ad[0:623], labels_test_ad[0:623], perturbation, 
                                        descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_ad

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ad
start= time.time()
print('RES --')
res_nn2_ad= stab.run_stability(nn2_model_ad, test_ad[0:623], labels_test_ad[0:623], 
                               descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_ad

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_ad
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_ad= stab.relative_stability(nn3_model_ad, test_ad[0:623], labels_test_ad[0:623], perturbation, 
                                        descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_ad

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ad
start= time.time()
print('RES --')
res_nn3_ad= stab.run_stability(nn3_model_ad, test_ad[0:623], labels_test_ad[0:623], 
                               descriptor_ad, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_ad

# 3. Chess (kr-vs-kp)

In [7]:
ch_path= 'data/chess/processed/'

train_ch= pd.read_csv(ch_path + 'X_train.csv')
test_ch = pd.read_csv(ch_path + 'X_test.csv')
labels_train_ch= pd.read_csv(ch_path + 'y_train.csv')
labels_test_ch = pd.read_csv(ch_path + 'y_test.csv')

In [ ]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_ch= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_ch.fit(train_ch, labels_train_ch.values.ravel())

acc_nn1_ch= sklearn.metrics.accuracy_score(labels_test_ch.values.ravel(), nn1_model_ch.predict(test_ch))
acc_nn1_ch

In [ ]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_ch= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_ch.fit(train_ch, labels_train_ch.values.ravel())

acc_nn2_ch= sklearn.metrics.accuracy_score(labels_test_ch.values.ravel(), nn2_model_ch.predict(test_ch))
acc_nn2_ch

In [ ]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ch= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_ch.fit(train_ch, labels_train_ch.values.ravel())

acc_nn3_ch= sklearn.metrics.accuracy_score(labels_test_ch.values.ravel(), nn3_model_ch.predict(test_ch))
acc_nn3_ch

In [ ]:
# definitions ---- 327 samples from test dataset

# get n and m parameters from train and labels_train
n_ch, m_ch= texp.get_n_m_sizes(test_ch.loc[0:326], labels_test_ch[0:326])

# conversion of train_ch and labels_train_ch data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_ch.values)
tn_lb_tr= torch.from_numpy(labels_train_ch.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_ch= dict()

h_min_dist_ch= texp.get_minimum_distance(train_ch)

# T-Exp explanation settings
descriptor_ch['h_min']= h_min_dist_ch
descriptor_ch['h_max']= 1
descriptor_ch['jacobian_eps']= 1e-3
descriptor_ch['max_itr']= 30
descriptor_ch['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_ch['num_samples']= 30
descriptor_ch['num_perts']= 10   # RIS/ROS
descriptor_ch['pert_max_distance']= (h_min_dist_ch/2)
descriptor_ch['num_runs']= 10    # RES
descriptor_ch['feature_metadata']= ['c'] * n_ch
descriptor_ch['p_norm']= 2
descriptor_ch['eps_norm']= 1e-6
descriptor_ch['top_k']= 0
descriptor_ch['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_ch['top_k'])

In [ ]:
# ---- 327 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_ch
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_ch= stab.relative_stability(nn1_model_ch, test_ch[0:326], labels_test_ch[0:326], perturbation, 
                                        descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_ch

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ch
start= time.time()
print('RES --')
res_nn1_ch= stab.run_stability(nn1_model_ch, test_ch[0:326], labels_test_ch[0:326], 
                               descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_ch

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_ch
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_ch= stab.relative_stability(nn2_model_ch, test_ch[0:326], labels_test_ch[0:326], perturbation, 
                                        descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_ch

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ch
start= time.time()
print('RES --')
res_nn2_ch= stab.run_stability(nn2_model_ch, test_ch[0:326], labels_test_ch[0:326], 
                               descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_ch

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_ch
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_ch= stab.relative_stability(nn3_model_ch, test_ch[0:326], labels_test_ch[0:326], perturbation, 
                                        descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_ch

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ch
start= time.time()
print('RES --')
res_nn3_ch= stab.run_stability(nn3_model_ch, test_ch[0:326], labels_test_ch[0:326], 
                               descriptor_ch, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_ch

# 4. COMPAS

In [8]:
cpas_path= 'data/compas/processed/'

train_cpas= pd.read_csv(cpas_path + 'X_train.csv')
test_cpas = pd.read_csv(cpas_path + 'X_test.csv')
labels_train_cpas= pd.read_csv(cpas_path + 'y_train.csv')
labels_test_cpas = pd.read_csv(cpas_path + 'y_test.csv')

In [ ]:
# 2024-05-30 04:34:50,307 Best: 0.729593 using {'batch_size': 32, 'lr': 0.01, 'max_epochs': 128, 
#                               'module__n_features': 13, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_cpas= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(16,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_cpas.fit(train_cpas, labels_train_cpas.values.ravel())

acc_nn1_cpas= sklearn.metrics.accuracy_score(labels_test_cpas.values.ravel(), nn1_model_cpas.predict(test_cpas))
acc_nn1_cpas

In [ ]:
# 2024-05-31 23:28:32,931 Best: 0.728194 using {'batch_size': 32, 'lr': 0.02, 'max_epochs': 32, 
#                               'module__n_features': 13, 'module__n_neurons': 32, 'module__nonlin': Tanh()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_cpas= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.02,
                            max_iter=32,
                            hidden_layer_sizes=(32, 32),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_cpas.fit(train_cpas, labels_train_cpas.values.ravel())

acc_nn2_cpas= sklearn.metrics.accuracy_score(labels_test_cpas.values.ravel(), nn2_model_cpas.predict(test_cpas))
acc_nn2_cpas

In [ ]:
# 2024-06-03 22:58:26,596 Best: 0.728264 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 32, 
#                               'module__n_features': 13, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_cpas= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=32,
                            hidden_layer_sizes=(16, 16, 16),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_cpas.fit(train_cpas, labels_train_cpas.values.ravel())

acc_nn3_cpas= sklearn.metrics.accuracy_score(labels_test_cpas.values.ravel(), nn3_model_cpas.predict(test_cpas))
acc_nn3_cpas

In [ ]:
# definitions ---- 409 samples from test dataset

# get n and m parameters from train and labels_train
n_cpas, m_cpas= texp.get_n_m_sizes(test_cpas.loc[0:408], labels_test_cpas[0:408])

# conversion of train_cpas and labels_train_cpas data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_cpas.values)
tn_lb_tr= torch.from_numpy(labels_train_cpas.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_cpas= dict()

h_min_dist_cpas= texp.get_minimum_distance(train_cpas)

# T-Exp explanation settings
descriptor_cpas['h_min']= h_min_dist_cpas
descriptor_cpas['h_max']= 1
descriptor_cpas['jacobian_eps']= 1e-3
descriptor_cpas['max_itr']= 30
descriptor_cpas['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_cpas['num_samples']= 30
descriptor_cpas['num_perts']= 10   # RIS/ROS
descriptor_cpas['pert_max_distance']= (h_min_dist_cpas/2)
descriptor_cpas['num_runs']= 10    # RES
descriptor_cpas['feature_metadata']= ['c'] * n_cpas
descriptor_cpas['p_norm']= 2
descriptor_cpas['eps_norm']= 1e-6
descriptor_cpas['top_k']= 0
descriptor_cpas['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_cpas['top_k'])

In [ ]:
# ---- 409 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_cpas
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_cpas= stab.relative_stability(nn1_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], perturbation, 
                                        descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_cpas

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_cpas
start= time.time()
print('RES --')
res_nn1_cpas= stab.run_stability(nn1_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], 
                               descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_cpas

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_cpas
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_cpas= stab.relative_stability(nn2_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], perturbation, 
                                        descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_cpas

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_cpas
start= time.time()
print('RES --')
res_nn2_cpas= stab.run_stability(nn2_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], 
                               descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_cpas

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_cpas
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_cpas= stab.relative_stability(nn3_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], perturbation, 
                                        descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_cpas

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_cpas
start= time.time()
print('RES --')
res_nn3_cpas= stab.run_stability(nn3_model_cpas, test_cpas[0:408], labels_test_cpas[0:408], 
                               descriptor_cpas, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_cpas

# 5. Diabetes

In [11]:
diab_path= 'data/diabetes/processed/'

train_diab= pd.read_csv(diab_path + 'X_train.csv')
test_diab = pd.read_csv(diab_path + 'X_test.csv')
labels_train_diab= pd.read_csv(diab_path + 'y_train.csv')
labels_test_diab = pd.read_csv(diab_path + 'y_test.csv')

In [12]:
# 2024-05-29 17:25:01,468 Best: 0.849094 using {'batch_size': 32, 'lr': 0.02, 'max_epochs': 128, 
#                               'module__n_features': 8, 'module__n_neurons': 64, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_diab= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.02,
                            max_iter=128,
                            hidden_layer_sizes=(64,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_diab.fit(train_diab, labels_train_diab.values.ravel())

acc_nn1_diab= sklearn.metrics.accuracy_score(labels_test_diab.values.ravel(), nn1_model_diab.predict(test_diab))
acc_nn1_diab

0.7532467532467533

In [13]:
# 2024-05-31 17:04:14,821 Best: 0.849508 using {'batch_size': 32, 'lr': 0.001, 'max_epochs': 128, 
#                               'module__n_features': 8, 'module__n_neurons': 128, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_diab= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.001,
                            max_iter=512,
                            hidden_layer_sizes=(128, 128),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_diab.fit(train_diab, labels_train_diab.values.ravel())

acc_nn2_diab= sklearn.metrics.accuracy_score(labels_test_diab.values.ravel(), nn2_model_diab.predict(test_diab))
acc_nn2_diab

0.7142857142857143

In [14]:
# 2024-06-03 16:05:00,462 Best: 0.852499 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 16, 
#                               'module__n_features': 8, 'module__n_neurons': 64, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_diab= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(64, 64, 64),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_diab.fit(train_diab, labels_train_diab.values.ravel())

acc_nn3_diab= sklearn.metrics.accuracy_score(labels_test_diab.values.ravel(), nn3_model_diab.predict(test_diab))
acc_nn3_diab

0.7597402597402597

In [15]:
# definitions ---- 126 samples from test dataset

# get n and m parameters from train and labels_train
n_diab, m_diab= texp.get_n_m_sizes(test_diab.loc[0:125], labels_test_diab[0:125])

# conversion of train_diab and labels_train_diab data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_diab.values)
tn_lb_tr= torch.from_numpy(labels_train_diab.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_diab= dict()

h_min_dist_diab= texp.get_minimum_distance(train_diab)

# T-Exp explanation settings
descriptor_diab['h_min']= h_min_dist_diab
descriptor_diab['h_max']= 1
descriptor_diab['jacobian_eps']= 1e-3
descriptor_diab['max_itr']= 30
descriptor_diab['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_diab['num_samples']= 30
descriptor_diab['num_perts']= 10   # RIS/ROS
descriptor_diab['pert_max_distance']= (h_min_dist_diab/2)
descriptor_diab['num_runs']= 10    # RES
descriptor_diab['feature_metadata']= ['c'] * n_diab
descriptor_diab['p_norm']= 2
descriptor_diab['eps_norm']= 1e-6
descriptor_diab['top_k']= 0
descriptor_diab['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_diab['top_k'])

In [34]:
# ---- 126 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_diab
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_diab= stab.relative_stability(nn1_model_diab, test_diab[0:125], labels_test_diab[0:125], perturbation, 
                                        descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_diab

RIS/ROS --
Progress: [████████████████████████████████████████] 125/125 Est wait 00:0.00


--- 2377.09 seconds ---


{'t_exp_ris_max': 195.6550581289991,
 'std(t_exp_ris_max)': 18.612012236226775,
 'shap_ris_max': 276.48585097826094,
 'std(shap_ris_max)': 45.26912986718888,
 'lime_ris_max': 29.683302166168474,
 'std(lime_ris_max)': 3.1903809053082277,
 't_exp_ris_mean': 2.9852187518273534,
 'std(t_exp_ris_mean)': 7.151608905753285,
 'shap_ris_mean': 12.210139594228275,
 'std(shap_ris_mean)': 19.348275222471152,
 'lime_ris_mean': 0.8017190869748956,
 'std(lime_ris_mean)': 1.8357353814908262,
 't_exp_ros_max': 16276.180169790043,
 'std(t_exp_ros_max)': 1490.4045193935083,
 'shap_ros_max': 146517.7583321755,
 'std(shap_ros_max)': 13082.259503765505,
 'lime_ros_max': 412.2534095766278,
 'std(lime_ros_max)': 57.66706982560556,
 't_exp_ros_mean': 43.31503376676555,
 'std(t_exp_ros_mean)': 230.4413964266719,
 'shap_ros_mean': 205.34509036431282,
 'std(shap_ros_mean)': 1349.899039541444,
 'lime_ros_mean': 4.8795164806377995,
 'std(lime_ros_mean)': 10.84592293485005,
 'shap_kernel_ris_max': 276.4858509786545,

In [36]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_diab
start= time.time()
print('RES --')
res_nn1_diab= stab.run_stability(nn1_model_diab, test_diab[0:125], labels_test_diab[0:125], 
                               descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_diab

RES --
Progress: [████████████████████████████████████████] 125/125 Est wait 00:0.09


--- 2115.93 seconds ---


{'texp_res': 9.490063172835475e-16,
 'shap_res': 2.2226980149236474e-16,
 'shap_kernel_res': 1.2566871346510768e-16,
 'shap_exact_res': 2.2226980149236474e-16,
 'lime_res': 6.397344083151129e-17,
 'itGd_res': 1.0310738718936457e-15,
 'iXGd_res': 5.4730396e-07,
 'dLif_res': 4.9623327e-07,
 'lwrp_res': 5.3312016e-07,
 'smoothG_res': 0.3579480709561981,
 'vanillaG_res': 1.2160663e-06,
 'GuidBprop_res': 1.2160663e-06,
 'occlusion_res': 5.462856e-07}

In [38]:
# evaluate relative input/output stability -- using the 2 first instances from train_diab
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_diab= stab.relative_stability(nn2_model_diab, test_diab[0:125], labels_test_diab[0:125], perturbation, 
                                        descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_diab

RIS/ROS --
Progress: [████████████████████████████████████████] 125/125 Est wait 00:0.03


--- 4279.75 seconds ---


{'t_exp_ris_max': 1434.1654305467005,
 'std(t_exp_ris_max)': 134.604068160892,
 'shap_ris_max': 9515.312719776788,
 'std(shap_ris_max)': 846.2347312327661,
 'lime_ris_max': 2042.1142326950405,
 'std(lime_ris_max)': 208.04105451001644,
 't_exp_ris_mean': 11.24293915498451,
 'std(t_exp_ris_mean)': 34.86560518217051,
 'shap_ris_mean': 39.53636871472338,
 'std(shap_ris_mean)': 265.02279050350063,
 'lime_ris_mean': 12.084823069212828,
 'std(lime_ris_mean)': 84.67702392838756,
 't_exp_ros_max': 5844.49583988835,
 'std(t_exp_ros_max)': 768.1205018921431,
 'shap_ros_max': 35084.53288140461,
 'std(shap_ros_max)': 3206.204151555847,
 'lime_ros_max': 148005.12348242916,
 'std(lime_ros_max)': 13181.675352057111,
 't_exp_ros_mean': 42.95683294556465,
 'std(t_exp_ros_mean)': 142.55204260830644,
 'shap_ros_mean': 155.398771090808,
 'std(shap_ros_mean)': 1095.5314269521489,
 'lime_ros_mean': 138.79018492769382,
 'std(lime_ros_mean)': 1455.4291099768136,
 'shap_kernel_ris_max': 349.9904315691398,
 'std

In [40]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_diab
start= time.time()
print('RES --')
res_nn2_diab= stab.run_stability(nn2_model_diab, test_diab[0:125], labels_test_diab[0:125], 
                               descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_diab

RES --
Progress: [████████████████████████████████████████] 125/125 Est wait 00:0.06


--- 3577.48 seconds ---


{'texp_res': 2.02900661633277e-15,
 'shap_res': 1.193812447098184e-16,
 'shap_kernel_res': 1.2719202621569003e-16,
 'shap_exact_res': 1.193812447098184e-16,
 'lime_res': 6.520806856012869e-17,
 'itGd_res': 2.589462819655575e-15,
 'iXGd_res': 1.9679126e-06,
 'dLif_res': 1.9678562e-06,
 'lwrp_res': 1.986778e-06,
 'smoothG_res': 1.5816260052017872,
 'vanillaG_res': 5.8788432e-06,
 'GuidBprop_res': 5.8788432e-06,
 'occlusion_res': 1.1920929e-06}

In [42]:
# evaluate relative input/output stability -- using the 2 first instances from train_diab
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_diab= stab.relative_stability(nn3_model_diab, test_diab[0:125], labels_test_diab[0:125], perturbation, 
                                        descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_diab

RIS/ROS --
Progress: [████████████████████████████████████████] 125/125 Est wait 00:0.00


--- 1998.67 seconds ---


{'t_exp_ris_max': 2823.6581468577974,
 'std(t_exp_ris_max)': 397.676252117684,
 'shap_ris_max': 1793.3153181951268,
 'std(shap_ris_max)': 202.83490289743625,
 'lime_ris_max': 54.796061172931296,
 'std(lime_ris_max)': 9.787480897059625,
 't_exp_ris_mean': 37.685526279656884,
 'std(t_exp_ris_mean)': 115.98038259283848,
 'shap_ris_mean': 28.922640964138697,
 'std(shap_ris_mean)': 63.81958868162377,
 'lime_ris_mean': 2.2587906045527366,
 'std(lime_ris_mean)': 3.5635607634815054,
 't_exp_ros_max': 3292407.4766008123,
 'std(t_exp_ros_max)': 295644.6814759473,
 'shap_ros_max': 611879.238628236,
 'std(shap_ros_max)': 55453.65082731761,
 'lime_ros_max': 99313.9059722736,
 'std(lime_ros_max)': 8901.826895678132,
 't_exp_ros_mean': 6603.090393776174,
 'std(t_exp_ros_mean)': 62514.739131786206,
 'shap_ros_mean': 1115.5826879081235,
 'std(shap_ros_mean)': 9094.606978374604,
 'lime_ros_mean': 177.32321264845783,
 'std(lime_ros_mean)': 1612.9854172496778,
 'shap_kernel_ris_max': 1793.315318193784,
 '

In [44]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_diab
start= time.time()
print('RES --')
res_nn3_diab= stab.run_stability(nn3_model_diab, test_diab[0:125], labels_test_diab[0:125], 
                               descriptor_diab, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_diab

RES --
Progress: [████████████████████████████████████████] 125/125 Est wait 00:0.03


--- 1791.36 seconds ---


{'texp_res': 5.024315033919233e-15,
 'shap_res': 1.2490611347050096e-16,
 'shap_kernel_res': 1.1775693440128312e-16,
 'shap_exact_res': 1.2490611347050096e-16,
 'lime_res': 4.4439065127221604e-17,
 'itGd_res': 5.402578481197087e-15,
 'iXGd_res': 4.2749457e-06,
 'dLif_res': 4.318047e-06,
 'lwrp_res': 4.268292e-06,
 'smoothG_res': 4.733137882473992,
 'vanillaG_res': 1.36295375e-05,
 'GuidBprop_res': 1.36295375e-05,
 'occlusion_res': 2.3856755e-06}

# 6. German Credit

In [34]:
ger_path= 'data/german/processed/'

train_ger= pd.read_csv(ger_path + 'X_train.csv')
test_ger = pd.read_csv(ger_path + 'X_test.csv')
labels_train_ger= pd.read_csv(ger_path + 'y_train.csv')
labels_test_ger = pd.read_csv(ger_path + 'y_test.csv')

In [40]:
# 2024-05-29 17:41:34,069 Best: 0.668612 using {'batch_size': 128, 'lr': 0.01, 'max_epochs': 128, 
#                               'module__n_features': 23, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_ger= MLPClassifier(batch_size= 128,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=512,
                            hidden_layer_sizes=(16,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_ger.fit(train_ger, labels_train_ger.values.ravel())

acc_nn1_ger= sklearn.metrics.accuracy_score(labels_test_ger.values.ravel(), nn1_model_ger.predict(test_ger))
acc_nn1_ger

0.63

In [45]:
# 2024-05-31 17:28:51,852 Best: 0.675973 using {'batch_size': 32, 'lr': 0.02, 'max_epochs': 16, 
#                               'module__n_features': 23, 'module__n_neurons': 256, 'module__nonlin': Tanh()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_ger= MLPClassifier(batch_size= 32,
                            learning_rate='constant', learning_rate_init= 0.02,
                            max_iter=32,
                            hidden_layer_sizes=(256, 256),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_ger.fit(train_ger, labels_train_ger.values.ravel())

acc_nn2_ger= sklearn.metrics.accuracy_score(labels_test_ger.values.ravel(), nn2_model_ger.predict(test_ger))
acc_nn2_ger

0.66

In [ ]:
# 2024-06-03 16:27:03,452 Best: 0.668002 using {'batch_size': 64, 'lr': 0.02, 'max_epochs': 32, 
#                               'module__n_features': 23, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ger= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.02,
                            max_iter=32,
                            hidden_layer_sizes=(16, 16, 16),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_ger.fit(train_ger, labels_train_ger.values.ravel())

acc_nn3_ger= sklearn.metrics.accuracy_score(labels_test_ger.values.ravel(), nn3_model_ger.predict(test_ger))
acc_nn3_ger

In [ ]:
# definitions ---- 154 samples from test dataset

# get n and m parameters from train and labels_train
n_ger, m_ger= texp.get_n_m_sizes(test_ger.loc[0:125], labels_test_ger[0:125])

# conversion of train_ger and labels_train_ger data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_ger.values)
tn_lb_tr= torch.from_numpy(labels_train_ger.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_ger= dict()

h_min_dist_ger= texp.get_minimum_distance(train_ger)

# T-Exp explanation settings
descriptor_ger['h_min']= h_min_dist_ger
descriptor_ger['h_max']= 1
descriptor_ger['jacobian_eps']= 1e-3
descriptor_ger['max_itr']= 30
descriptor_ger['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_ger['num_samples']= 30
descriptor_ger['num_perts']= 10   # RIS/ROS
descriptor_ger['pert_max_distance']= (h_min_dist_ger/2)
descriptor_ger['num_runs']= 10    # RES
descriptor_ger['feature_metadata']= ['c'] * n_ger
descriptor_ger['p_norm']= 2
descriptor_ger['eps_norm']= 1e-6
descriptor_ger['top_k']= 0
descriptor_ger['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_ger['top_k'])

In [ ]:
# ---- 154 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_ger
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_ger= stab.relative_stability(nn1_model_ger, test_ger[0:125], labels_test_ger[0:125], perturbation, 
                                        descriptor_ger, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_ger

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ger
start= time.time()
print('RES --')
res_nn1_ger= stab.run_stability(nn1_model_ger, test_ger[0:125], labels_test_ger[0:125], 
                               descriptor_ger, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_ger

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_ger
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_ger= stab.relative_stability(nn2_model_ger, test_ger[0:125], labels_test_ger[0:125], perturbation, 
                                        descriptor_ger, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_ger

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ger
start= time.time()
print('RES --')
res_nn2_ger= stab.run_stability(nn2_model_ger, test_ger[0:125], labels_test_ger[0:125], 
                               descriptor_ger, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_ger

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_ger
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_ger= stab.relative_stability(nn3_model_ger, test_ger[0:125], labels_test_ger[0:125], perturbation, 
                                        descriptor_ger, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_ger

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ger
start= time.time()
print('RES --')
res_nn3_ger= stab.run_stability(nn3_model_ger, test_ger[0:125], labels_test_ger[0:125], 
                               descriptor_ger, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_ger

# 7. HELOC

In [11]:
hel_path= 'data/heloc/processed/'

train_hel= pd.read_csv(hel_path + 'X_train.csv')
test_hel = pd.read_csv(hel_path + 'X_test.csv')
labels_train_hel= pd.read_csv(hel_path + 'y_train.csv')
labels_test_hel = pd.read_csv(hel_path + 'y_test.csv')

In [ ]:
# 2024-05-30 09:37:27,988 Best: 0.802750 using {'batch_size': 16, 'lr': 0.001, 'max_epochs': 128, 
#                               'module__n_features': 37, 'module__n_neurons': 256, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_hel= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.001,
                            max_iter=128,
                            hidden_layer_sizes=(256,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_hel.fit(train_hel, labels_train_hel.values.ravel())

acc_nn1_hel= sklearn.metrics.accuracy_score(labels_test_hel.values.ravel(), nn1_model_hel.predict(test_hel))
acc_nn1_hel

In [ ]:
# 2024-06-01 04:05:45,253 Best: 0.802888 using {'batch_size': 16, 'lr': 0.001, 'max_epochs': 128, 
#                               'module__n_features': 37, 'module__n_neurons': 16, 'module__nonlin': Tanh()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_hel= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.001,
                            max_iter=128,
                            hidden_layer_sizes=(16, 16),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_hel.fit(train_hel, labels_train_hel.values.ravel())

acc_nn2_hel= sklearn.metrics.accuracy_score(labels_test_hel.values.ravel(), nn2_model_hel.predict(test_hel))
acc_nn2_hel

In [ ]:
# 2024-06-04 04:15:54,809 Best: 0.802922 using {'batch_size': 64, 'lr': 0.001, 'max_epochs': 16, 
#                               'module__n_features': 37, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_hel= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.001,
                            max_iter=16,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_hel.fit(train_hel, labels_train_hel.values.ravel())

acc_nn3_hel= sklearn.metrics.accuracy_score(labels_test_hel.values.ravel(), nn3_model_hel.predict(test_hel))
acc_nn3_hel

In [ ]:
# definitions ---- 499 samples from test dataset

# get n and m parameters from train and labels_train
n_hel, m_hel= texp.get_n_m_sizes(test_hel.loc[0:498], labels_test_hel[0:498])

# conversion of train_hel and labels_train_hel data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_hel.values)
tn_lb_tr= torch.from_numpy(labels_train_hel.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_hel= dict()

h_min_dist_hel= texp.get_minimum_distance(train_hel)

# T-Exp explanation settings
descriptor_hel['h_min']= h_min_dist_hel
descriptor_hel['h_max']= 1
descriptor_hel['jacobian_eps']= 1e-3
descriptor_hel['max_itr']= 30
descriptor_hel['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_hel['num_samples']= 30
descriptor_hel['num_perts']= 10   # RIS/ROS
descriptor_hel['pert_max_distance']= (h_min_dist_hel/2)
descriptor_hel['num_runs']= 10    # RES
descriptor_hel['feature_metadata']= ['c'] * n_hel
descriptor_hel['p_norm']= 2
descriptor_hel['eps_norm']= 1e-6
descriptor_hel['top_k']= 0
descriptor_hel['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_hel['top_k'])

In [ ]:
# ---- 499 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_hel
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_hel= stab.relative_stability(nn1_model_hel, test_hel[0:498], labels_test_hel[0:498], perturbation, 
                                        descriptor_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_hel

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_hel
start= time.time()
print('RES --')
res_nn1_hel= stab.run_stability(nn1_model_hel, test_hel[0:498], labels_test_hel[0:498], 
                               descriptor_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_hel

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_hel
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_hel= stab.relative_stability(nn2_model_hel, test_hel[0:498], labels_test_hel[0:498], perturbation, 
                                        descriptor_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_hel

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_hel
start= time.time()
print('RES --')
res_nn2_hel= stab.run_stability(nn2_model_hel, test_hel[0:498], labels_test_hel[0:498], 
                               descriptor_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_hel

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_hel
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_hel= stab.relative_stability(nn3_model_hel, test_hel[0:498], labels_test_hel[0:498], perturbation, 
                                        descriptor_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_hel

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_hel
start= time.time()
print('RES --')
res_nn3_hel= stab.run_stability(nn3_model_hel, test_hel[0:498], labels_test_hel[0:498], 
                               descriptor_hel, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_hel

# 8. HIGGS

In [12]:
hig_path= 'data/higgs/processed/'

train_hig= pd.read_csv(hig_path + 'X_train.csv')
test_hig = pd.read_csv(hig_path + 'X_test.csv')
labels_train_hig= pd.read_csv(hig_path + 'y_train.csv')
labels_test_hig = pd.read_csv(hig_path + 'y_test.csv')

In [ ]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_hig= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_hig.fit(train_hig, labels_train_hig.values.ravel())

acc_nn1_hig= sklearn.metrics.accuracy_score(labels_test_hig.values.ravel(), nn1_model_hig.predict(test_hig))
acc_nn1_hig

In [ ]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_hig= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_hig.fit(train_hig, labels_train_hig.values.ravel())

acc_nn2_hig= sklearn.metrics.accuracy_score(labels_test_hig.values.ravel(), nn2_model_hig.predict(test_hig))
acc_nn2_hig

In [ ]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_hig= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_hig.fit(train_hig, labels_train_hig.values.ravel())

acc_nn3_hig= sklearn.metrics.accuracy_score(labels_test_hig.values.ravel(), nn3_model_hig.predict(test_hig))
acc_nn3_hig

In [ ]:
# definitions ---- 644 samples from test dataset

# get n and m parameters from train and labels_train
n_hig, m_hig= texp.get_n_m_sizes(test_hig.loc[0:643], labels_test_hig[0:643])

# conversion of train_hig and labels_train_hig data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_hig.values)
tn_lb_tr= torch.from_numpy(labels_train_hig.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_hig= dict()

h_min_dist_hig= texp.get_minimum_distance(train_hig)

# T-Exp explanation settings
descriptor_hig['h_min']= h_min_dist_hig
descriptor_hig['h_max']= 1
descriptor_hig['jacobian_eps']= 1e-3
descriptor_hig['max_itr']= 30
descriptor_hig['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_hig['num_samples']= 30
descriptor_hig['num_perts']= 10   # RIS/ROS
descriptor_hig['pert_max_distance']= (h_min_dist_hig/2)
descriptor_hig['num_runs']= 10    # RES
descriptor_hig['feature_metadata']= ['c'] * n_hig
descriptor_hig['p_norm']= 2
descriptor_hig['eps_norm']= 1e-6
descriptor_hig['top_k']= 0
descriptor_hig['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_hig['top_k'])

In [ ]:
# ---- 644 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_hig
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_hig= stab.relative_stability(nn1_model_hig, test_hig[0:643], labels_test_hig[0:643], perturbation, 
                                        descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_hig

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_hig
start= time.time()
print('RES --')
res_nn1_hig= stab.run_stability(nn1_model_hig, test_hig[0:643], labels_test_hig[0:643], 
                               descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_hig

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_hig
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_hig= stab.relative_stability(nn2_model_hig, test_hig[0:643], labels_test_hig[0:643], perturbation, 
                                        descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_hig

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_hig
start= time.time()
print('RES --')
res_nn2_hig= stab.run_stability(nn2_model_hig, test_hig[0:643], labels_test_hig[0:643], 
                               descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_hig

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_hig
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_hig= stab.relative_stability(nn3_model_hig, test_hig[0:643], labels_test_hig[0:643], perturbation, 
                                        descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_hig

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_hig
start= time.time()
print('RES --')
res_nn3_hig= stab.run_stability(nn3_model_hig, test_hig[0:643], labels_test_hig[0:643], 
                               descriptor_hig, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_hig

# 9. Independent

In [16]:
indep_path= 'data/independent/processed/'

train_indep= pd.read_csv(indep_path + 'X_train.csv')
test_indep = pd.read_csv(indep_path + 'X_test.csv')
labels_train_indep= pd.read_csv(indep_path + 'y_train.csv')
labels_test_indep = pd.read_csv(indep_path + 'y_test.csv')

In [17]:
# 2024-05-29 17:02:04,589 Best: 0.997288 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 32, 
#                               'module__n_features': 6, 'module__n_neurons': 256, 'module__nonlin': Tanh()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_indep= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=64,
                            hidden_layer_sizes=(256,),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_indep.fit(train_indep, labels_train_indep.values.ravel())

acc_nn1_indep= sklearn.metrics.accuracy_score(labels_test_indep.values.ravel(), nn1_model_indep.predict(test_indep))
acc_nn1_indep

1.0

In [18]:
# 2024-05-31 16:17:24,671 Best: 0.997288 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 16, 
#                               'module__n_features': 6, 'module__n_neurons': 32, 'module__nonlin': Tanh()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_indep= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=64,
                            hidden_layer_sizes=(32, 32),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_indep.fit(train_indep, labels_train_indep.values.ravel())

acc_nn2_indep= sklearn.metrics.accuracy_score(labels_test_indep.values.ravel(), nn2_model_indep.predict(test_indep))
acc_nn2_indep

0.9666666666666667

In [19]:
# 2024-06-03 15:02:42,848 Best: 0.997667 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 16, 
#                               'module__n_features': 6, 'module__n_neurons': 32, 'module__nonlin': Tanh()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_indep= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=64,
                            hidden_layer_sizes=(32, 32, 32),
                            activation='tanh',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_indep.fit(train_indep, labels_train_indep.values.ravel())

acc_nn3_indep= sklearn.metrics.accuracy_score(labels_test_indep.values.ravel(), nn3_model_indep.predict(test_indep))
acc_nn3_indep

1.0

In [20]:
# definitions ---- 60 samples from test dataset

# get n and m parameters from train and labels_train
n_indep, m_indep= texp.get_n_m_sizes(test_indep, labels_test_indep)

# conversion of train_indep and labels_train_indep data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_indep.values)
tn_lb_tr= torch.from_numpy(labels_train_indep.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_indep= dict()

h_min_dist_indep= texp.get_minimum_distance(train_indep)

# T-Exp explanation settings
descriptor_indep['h_min']= h_min_dist_indep
descriptor_indep['h_max']= 1
descriptor_indep['jacobian_eps']= 1e-3
descriptor_indep['max_itr']= 30
descriptor_indep['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_indep['num_samples']= 30
descriptor_indep['num_perts']= 10   # RIS/ROS
descriptor_indep['pert_max_distance']= (h_min_dist_indep/2)
descriptor_indep['num_runs']= 10    # RES
descriptor_indep['feature_metadata']= ['c'] * n_indep
descriptor_indep['p_norm']= 2
descriptor_indep['eps_norm']= 1e-6
descriptor_indep['top_k']= 0
descriptor_indep['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_indep['top_k'])

In [21]:
# ---- 60 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_indep
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_indep= stab.relative_stability(nn1_model_indep, test_indep, labels_test_indep, perturbation, 
                                        descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_indep

RIS/ROS --
Progress: [████████████████████████████████████████] 60/60 Est wait 00:0.03


--- 737.44 seconds ---


{'t_exp_ris_max': 90.38846388392334,
 'std(t_exp_ris_max)': 12.622843990284167,
 'shap_ris_max': 191433.5913678973,
 'std(shap_ris_max)': 24503.14049117982,
 'lime_ris_max': 20.685320287567077,
 'std(lime_ris_max)': 2.9540460639200976,
 't_exp_ris_mean': 2.737558946702974,
 'std(t_exp_ris_mean)': 4.378116438302378,
 'shap_ris_mean': 929.7750891665377,
 'std(shap_ris_mean)': 7046.57129256637,
 'lime_ris_mean': 1.0818994831381867,
 'std(lime_ris_mean)': 1.325033327944721,
 't_exp_ros_max': 70165.41694342354,
 'std(t_exp_ros_max)': 10005.441453518339,
 'shap_ros_max': 906705.0771769545,
 'std(shap_ros_max)': 117354.19522732789,
 'lime_ros_max': 23571.862919442992,
 'std(lime_ros_max)': 3866.7764141122666,
 't_exp_ros_mean': 905.9625302894107,
 'std(t_exp_ros_mean)': 4247.042927598499,
 'shap_ros_mean': 2729.892686251868,
 'std(shap_ros_mean)': 14783.735337385366,
 'lime_ros_mean': 208.42727522570735,
 'std(lime_ros_mean)': 895.4109543459091,
 'shap_kernel_ris_max': 275.30872813452896,
 's

In [24]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_indep
start= time.time()
print('RES --')
res_nn1_indep= stab.run_stability(nn1_model_indep, test_indep, labels_test_indep, 
                               descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_indep

RES --
Progress: [████████████████████████████████████████] 60/60 Est wait 00:0.05


--- 631.13 seconds ---


{'texp_res': 5.6217203786222585e-15,
 'shap_res': 1.3019674641152052e-16,
 'shap_kernel_res': 1.2719202621569003e-16,
 'shap_exact_res': 1.3019674641152052e-16,
 'lime_res': 8.328592404739402e-17,
 'itGd_res': 2.6651134375294657e-15,
 'iXGd_res': 1.652098e-06,
 'dLif_res': 1.652098e-06,
 'lwrp_res': 1.508186e-06,
 'smoothG_res': 0.7889713025739256,
 'vanillaG_res': 2.8622644e-06,
 'GuidBprop_res': 2.8622644e-06,
 'occlusion_res': 1.7192608e-06}

In [26]:
# evaluate relative input/output stability -- using the 2 first instances from train_indep
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_indep= stab.relative_stability(nn2_model_indep, test_indep, labels_test_indep, perturbation, 
                                        descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_indep

RIS/ROS --
Progress: [████████████████████████████████████████] 60/60 Est wait 00:0.03


--- 675.51 seconds ---


{'t_exp_ris_max': 56.896366142029486,
 'std(t_exp_ris_max)': 8.189184948883648,
 'shap_ris_max': 16276.716626165067,
 'std(shap_ris_max)': 3487.1931363636795,
 'lime_ris_max': 461.9796648119015,
 'std(lime_ris_max)': 59.39339886247817,
 't_exp_ris_mean': 2.8970697751160346,
 'std(t_exp_ris_mean)': 3.318017233223311,
 'shap_ris_mean': 402.3956111738581,
 'std(shap_ris_mean)': 1272.4295241038471,
 'lime_ris_mean': 6.201296171292145,
 'std(lime_ris_mean)': 22.96186682806372,
 't_exp_ros_max': 10702.898070391842,
 'std(t_exp_ros_max)': 1436.6236441157057,
 'shap_ros_max': 8080972.748554485,
 'std(shap_ros_max)': 1034379.5318456899,
 'lime_ros_max': 15198.306452736642,
 'std(lime_ros_max)': 2095.3656873476402,
 't_exp_ros_mean': 39.95999750424293,
 'std(t_exp_ros_mean)': 146.94782444092803,
 'shap_ros_mean': 14930.038264451527,
 'std(shap_ros_mean)': 104960.77243758597,
 'lime_ros_mean': 75.3056909727184,
 'std(lime_ros_mean)': 266.4197360343594,
 'shap_kernel_ris_max': 245.25287930968213,


In [28]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_indep
start= time.time()
print('RES --')
res_nn2_indep= stab.run_stability(nn2_model_indep, test_indep, labels_test_indep, 
                               descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_indep

RES --
Progress: [████████████████████████████████████████] 60/60 Est wait 00:0.04


--- 581.22 seconds ---


{'texp_res': 5.17894423319163e-15,
 'shap_res': 1.1102230246251565e-16,
 'shap_kernel_res': 1.1775693440128312e-16,
 'shap_exact_res': 1.1102230246251565e-16,
 'lime_res': 8.326785621649077e-17,
 'itGd_res': 2.083194168503685e-15,
 'iXGd_res': 1.3486991e-06,
 'dLif_res': 1.3486991e-06,
 'lwrp_res': 1.5078915e-06,
 'smoothG_res': 0.622871692643357,
 'vanillaG_res': 2.3511748e-06,
 'GuidBprop_res': 2.3511748e-06,
 'occlusion_res': 1.4354699e-06}

In [30]:
# evaluate relative input/output stability -- using the 2 first instances from train_indep
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_indep= stab.relative_stability(nn3_model_indep, test_indep, labels_test_indep, perturbation, 
                                        descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_indep

RIS/ROS --
Progress: [████████████████████████████████████████] 60/60 Est wait 00:0.04


--- 743.43 seconds ---


{'t_exp_ris_max': 291.90093659652456,
 'std(t_exp_ris_max)': 38.86135910480629,
 'shap_ris_max': 12049.986135231107,
 'std(shap_ris_max)': 1996.711025288372,
 'lime_ris_max': 181.1842866676077,
 'std(lime_ris_max)': 27.381876530963797,
 't_exp_ris_mean': 8.429238375879518,
 'std(t_exp_ris_mean)': 16.764960969963962,
 'shap_ris_mean': 165.25166230879114,
 'std(shap_ris_mean)': 491.45947129546073,
 'lime_ris_mean': 5.551063997597867,
 'std(lime_ris_mean)': 11.447876241394065,
 't_exp_ros_max': 4099.889309498876,
 'std(t_exp_ros_max)': 672.9647819895787,
 'shap_ros_max': 468740.1445592275,
 'std(shap_ros_max)': 60212.41428286577,
 'lime_ros_max': 4558.219426367684,
 'std(lime_ros_max)': 582.3169368710138,
 't_exp_ros_mean': 31.28080735289412,
 'std(t_exp_ros_mean)': 145.50884767797007,
 'shap_ros_mean': 1349.3569334257556,
 'std(shap_ros_mean)': 6792.216271164167,
 'lime_ros_mean': 12.573008620450176,
 'std(lime_ros_mean)': 59.042743302444656,
 'shap_kernel_ris_max': 926.0059203779998,
 '

In [32]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_indep
start= time.time()
print('RES --')
res_nn3_indep= stab.run_stability(nn3_model_indep, test_indep, labels_test_indep, 
                               descriptor_indep, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_indep

RES --
Progress: [████████████████████████████████████████] 60/60 Est wait 00:0.02


--- 669.93 seconds ---


{'texp_res': 6.158736452937403e-15,
 'shap_res': 1.2719202621569003e-16,
 'shap_kernel_res': 1.1775693440128312e-16,
 'shap_exact_res': 1.2719202621569003e-16,
 'lime_res': 8.777083671441753e-17,
 'itGd_res': 1.8477795749834654e-15,
 'iXGd_res': 1.1740754e-06,
 'dLif_res': 1.1935821e-06,
 'lwrp_res': 1.3513308e-06,
 'smoothG_res': 0.7010142167547295,
 'vanillaG_res': 2.9103078e-06,
 'GuidBprop_res': 2.9103078e-06,
 'occlusion_res': 1.2686131e-06}

# 10. LSA - Law School Admission

In [14]:
lsa_path= 'data/law_school_admission/processed/'

train_lsa= pd.read_csv(lsa_path + 'X_train.csv')
test_lsa = pd.read_csv(lsa_path + 'X_test.csv')
labels_train_lsa= pd.read_csv(lsa_path + 'y_train.csv')
labels_test_lsa = pd.read_csv(lsa_path + 'y_test.csv')

In [ ]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_lsa= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_lsa.fit(train_lsa, labels_train_lsa.values.ravel())

acc_nn1_lsa= sklearn.metrics.accuracy_score(labels_test_lsa.values.ravel(), nn1_model_lsa.predict(test_lsa))
acc_nn1_lsa

In [ ]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_lsa= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_lsa.fit(train_lsa, labels_train_lsa.values.ravel())

acc_nn2_lsa= sklearn.metrics.accuracy_score(labels_test_lsa.values.ravel(), nn2_model_lsa.predict(test_lsa))
acc_nn2_lsa

In [ ]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_lsa= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_lsa.fit(train_lsa, labels_train_lsa.values.ravel())

acc_nn3_lsa= sklearn.metrics.accuracy_score(labels_test_lsa.values.ravel(), nn3_model_lsa.predict(test_lsa))
acc_nn3_lsa

In [ ]:
# definitions ---- 574 samples from test dataset

# get n and m parameters from train and labels_train
n_lsa, m_lsa= texp.get_n_m_sizes(test_lsa.loc[0:573], labels_test_lsa[0:573])

# conversion of train_lsa and labels_train_lsa data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_lsa.values)
tn_lb_tr= torch.from_numpy(labels_train_lsa.values.ravel().astype(int))


# define a descriptor to synthetic data
descriptor_lsa= dict()

h_min_dist_lsa= texp.get_minimum_distance(train_lsa)

# T-Exp explanation settings
descriptor_lsa['h_min']= h_min_dist_lsa
descriptor_lsa['h_max']= 1
descriptor_lsa['jacobian_eps']= 1e-3
descriptor_lsa['max_itr']= 30
descriptor_lsa['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_lsa['num_samples']= 30
descriptor_lsa['num_perts']= 10   # RIS/ROS
descriptor_lsa['pert_max_distance']= (h_min_dist_lsa/2)
descriptor_lsa['num_runs']= 10    # RES
descriptor_lsa['feature_metadata']= ['c'] * n_lsa
descriptor_lsa['p_norm']= 2
descriptor_lsa['eps_norm']= 1e-6
descriptor_lsa['top_k']= 0
descriptor_lsa['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_lsa['top_k'])

In [ ]:
# ---- 574 samples from test dataset

# evaluate relative input/output stability -- using the 2 first instances from train_lsa
start= time.time()
print('RIS/ROS --') 
ris_ros_nn1_lsa= stab.relative_stability(nn1_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], perturbation, 
                                        descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn1_lsa

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_lsa
start= time.time()
print('RES --')
res_nn1_lsa= stab.run_stability(nn1_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], 
                               descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn1_lsa

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_lsa
start= time.time()
print('RIS/ROS --') 
ris_ros_nn2_lsa= stab.relative_stability(nn2_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], perturbation, 
                                        descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn2_lsa

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_lsa
start= time.time()
print('RES --')
res_nn2_lsa= stab.run_stability(nn2_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], 
                               descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn2_lsa

In [ ]:
# evaluate relative input/output stability -- using the 2 first instances from train_lsa
start= time.time()
print('RIS/ROS --') 
ris_ros_nn3_lsa= stab.relative_stability(nn3_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], perturbation, 
                                        descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

ris_ros_nn3_lsa

In [ ]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_lsa
start= time.time()
print('RES --')
res_nn3_lsa= stab.run_stability(nn3_model_lsa, test_lsa[0:573], labels_test_lsa[0:573], 
                               descriptor_lsa, is_model_NN=True)

print("\n--- %s seconds ---" % np.round((time.time()- start), 2))

res_nn3_lsa